# El bucle con verificador: optimizar código probando, midiendo y quedándote con lo mejor

> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

**Facsímil 5 · Agentes y orquestación** — capítulo 11 (el bucle con verificador; el LLM como optimizador).

La idea más potente de los agentes que optimizan código no es que el modelo sea «muy listo». Es el **bucle**:
proponer una transformación, **comprobar que sigue siendo correcta**, **medir si de verdad es más rápida** y
quedarse con la mejor; repetir. El modelo aporta las ideas, pero quien manda es el **verificador**, porque
una idea brillante que da resultados equivocados no vale nada.

En este cuaderno construyes ese bucle **de verdad**, sin LLM y sin compilador: un mini *auto-scheduler* que
optimiza una operación real —**multiplicar dos matrices**— probando distintas implementaciones, verificando
su corrección contra numpy y **cronometrando el tiempo real** en tu máquina. La política que propone empieza
siendo ciega (aleatoria), igual que un modelo que adivina; al final verás que **cambiar esa política aleatoria
por un LLM que lee el feedback es exactamente lo que hace un sistema como COMPILOT**.

### Qué vas a aprender
- Qué es un **verificador**: corrección (¿da el resultado bueno?) y rendimiento (¿es de verdad más rápido?).
- Por qué una transformación que parece una optimización puede ser **incorrecta**, y cómo el verificador la **rechaza**.
- El **bucle del agente**: proponer, juzgar, guardar la mejor, repetir; y cómo el mejor **sube y se estanca**.
- Por qué **best-of-N** (probar varias veces y quedarse con la mejor) supera a una sola tirada.
- Cómo resumir mejoras con la **media geométrica** sobre varios tamaños de problema.

### Cómo se usa
Las celdas vienen **sin ejecutar**: pulsa el play de arriba abajo. Todas las cifras salen de cronometrar
**tu** ordenador, así que **variarán** de una máquina a otra y de una ejecución a otra. No te fijes en el
número exacto, fíjate en la **tendencia**: el mejor mejora varias veces respecto a la base, y best-of-N
gana a la tirada única.

### Cuánto cuesta
Unos 15 minutos. Solo CPU; sin GPU ni claves. Únicamente numpy y matplotlib (ya vienen en Colab).

In [ ]:
# Colab ya los trae. En tu maquina:  pip install numpy matplotlib
import time
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

def medir_tiempo(fn, reps=3):
    'Ejecuta fn() varias veces y devuelve el MEJOR tiempo (el minimo, el menos contaminado por ruido).'
    fn()  # calentamiento: descartamos la primera, que suele incluir costes unicos
    mejor = float("inf")
    for _ in range(reps):
        t0 = time.perf_counter()
        fn()
        t1 = time.perf_counter()
        mejor = min(mejor, t1 - t0)
    return mejor

print("Listo. Vamos a optimizar una multiplicacion de matrices.")

## 1. El problema: multiplicar dos matrices

Multiplicar matrices $C = A \times B$ es una de las operaciones más comunes del cómputo numérico (está
debajo de casi todas las redes neuronales). La definición es un triple bucle: cada celda de $C$ es la
suma de productos de una fila de $A$ por una columna de $B$.

$$C_{ij} = \sum_{k} A_{ik}\,B_{kj}$$

Escrita tal cual en **Python puro**, esa triple suma es nuestra implementación **base**: correcta pero
lenta. Será el punto de partida (*speedup* = 1) contra el que comparemos todo lo demás. La **referencia
de corrección** será siempre `A @ B` de numpy, que sabemos que está bien.

In [ ]:
def matmul_base(A, B):
    'Multiplicacion de matrices con el triple bucle clasico, en Python puro. Correcta pero lenta.'
    N = A.shape[0]
    A_l = A.tolist()           # pasamos a listas de Python: indexar numpy elemento a elemento es lentisimo
    B_l = B.tolist()
    C = [[0.0] * N for _ in range(N)]
    for i in range(N):
        Ai = A_l[i]
        Ci = C[i]
        for j in range(N):
            s = 0.0
            for k in range(N):
                s += Ai[k] * B_l[k][j]
            Ci[j] = s
    return np.array(C)

N = 64
A = np.random.randn(N, N)
B = np.random.randn(N, N)
referencia = A @ B                       # la verdad: numpy lo hace bien

C_base = matmul_base(A, B)
print("La base coincide con numpy:", np.allclose(C_base, referencia))
print(f"Tiempo de la base (Python puro): {medir_tiempo(lambda: matmul_base(A, B)):.4f} s")
print(f"Tiempo de numpy A@B:             {medir_tiempo(lambda: A @ B):.6f} s")

La base es **correcta** (coincide con numpy) pero **mucho más lenta**: numpy delega la operación a código
compilado y altamente optimizado, mientras que nuestro triple bucle interpreta cada multiplicación en
Python. Ahí está el margen de mejora que el agente va a buscar. Pero ojo: ese margen solo cuenta **si la
transformación sigue dando el resultado correcto**. De eso se encarga el verificador.

## 2. El verificador: corrección primero, velocidad después

El verificador es el corazón de todo. Recibe una implementación candidata y hace **dos cosas en este orden**:

1. **Corrección.** Ejecuta el candidato y compara su resultado con la referencia de numpy (`np.allclose`).
   Si no coincide, la transformación es **ilegal** y se rechaza sin más: no nos interesa lo rápido que sea
   algo que calcula mal.
2. **Rendimiento.** Solo si es correcto, lo cronometra (varias repeticiones, nos quedamos con el mínimo) y
   calcula el **speedup** respecto a la base: cuántas veces más rápido es.

Fíjate en que el verificador **no se fía** de quién propuso el candidato ni de lo prometedor que parezca:
lo comprueba con números. Esa desconfianza sistemática es lo que hace que el bucle sea de fiar.

In [ ]:
def verificador(nombre, ejecuta, referencia, tiempo_base, reps=3):
    'Juzga un candidato: primero corrige, luego cronometra. Devuelve un informe.'
    C = ejecuta()
    correcto = (C.shape == referencia.shape) and np.allclose(C, referencia)
    if not correcto:
        return {"nombre": nombre, "correcto": False, "tiempo": None, "speedup": None}
    t = medir_tiempo(ejecuta, reps)
    return {"nombre": nombre, "correcto": True, "tiempo": t,
            "speedup": tiempo_base / t if tiempo_base else None}

# Medimos la base una vez: es la vara de medir de todos los demas.
tiempo_base = medir_tiempo(lambda: matmul_base(A, B))

# Probamos el verificador con un candidato que sabemos correcto: el propio numpy.
informe = verificador("numpy A@B", lambda: A @ B, referencia, tiempo_base)
print(informe)
print(f"\nnumpy es ~{informe['speedup']:.0f}x mas rapido que la base en esta maquina.")

## 3. El espacio de búsqueda: el catálogo de transformaciones

El agente no inventa código de la nada: explora un **espacio de transformaciones** conocidas de la misma
operación. Las nuestras juegan con dos palancas clásicas de cualquier compilador:

- **Orden de los bucles** (`ijk` frente a `ikj`): cambia cómo se recorre la memoria.
- **Bloques o *tiling*** (`bloques b=...`): trocear las matrices en bloques de tamaño `b` y multiplicar
  bloque a bloque con numpy. Bloques pequeños dejan casi todo el trabajo en Python (lento); bloques grandes
  delegan más en numpy (rápido).
- **Vectorización**: desde procesar fila a fila con numpy hasta delegar la operación entera (`A @ B`).

Y, a propósito, metemos en el catálogo **dos «optimizaciones» tentadoras pero incorrectas**, para ver al
verificador en acción. Construimos el catálogo como una lista de candidatos; cada uno sabe ejecutarse.

In [ ]:
def construye_catalogo(A, B):
    'Devuelve la lista de candidatos (transformaciones) de la misma multiplicacion de matrices.'
    N = A.shape[0]
    cat = []

    cat.append({"nombre": "bucle ijk (base, Python puro)",
                "ejecuta": (lambda: matmul_base(A, B)), "esperado": "correcta"})

    def matmul_ikj():                       # mismo calculo, bucles reordenados a i-k-j
        A_l = A.tolist(); B_l = B.tolist()
        C = [[0.0] * N for _ in range(N)]
        for i in range(N):
            Ai = A_l[i]; Ci = C[i]
            for k in range(N):
                aik = Ai[k]; Bk = B_l[k]
                for j in range(N):
                    Ci[j] += aik * Bk[j]
            # acumula sobre j con la fila k entera: mejor localidad
        return np.array(C)
    cat.append({"nombre": "bucle ikj (Python puro)",
                "ejecuta": matmul_ikj, "esperado": "correcta"})

    for b in [2, 4, 8, 16, 32]:             # tiling: bloques de bxb multiplicados con numpy
        def por_bloques(b=b):
            C = np.zeros((N, N))
            for i0 in range(0, N, b):
                for j0 in range(0, N, b):
                    acc = C[i0:i0 + b, j0:j0 + b]          # vista: el += modifica C en su sitio
                    for k0 in range(0, N, b):
                        acc += A[i0:i0 + b, k0:k0 + b] @ B[k0:k0 + b, j0:j0 + b]
            return C
        cat.append({"nombre": f"bloques numpy b={b}",
                    "ejecuta": por_bloques, "esperado": "correcta"})

    def por_filas():                         # cada fila de C de un tiron con numpy
        C = np.empty((N, N))
        for i in range(N):
            C[i] = A[i] @ B
        return C
    cat.append({"nombre": "filas vectorizadas (numpy)",
                "ejecuta": por_filas, "esperado": "correcta"})

    cat.append({"nombre": "numpy A@B (todo vectorizado)",
                "ejecuta": (lambda: A @ B), "esperado": "correcta"})

    # --- dos optimizaciones TENTADORAS pero INCORRECTAS ---
    cat.append({"nombre": "A*B elementwise (incorrecta)",
                "ejecuta": (lambda: A * B), "esperado": "incorrecta"})
    cat.append({"nombre": "B@A operandos cambiados (incorrecta)",
                "ejecuta": (lambda: B @ A), "esperado": "incorrecta"})
    return cat

catalogo = construye_catalogo(A, B)
print(f"El catalogo tiene {len(catalogo)} candidatos:")
for c in catalogo:
    print("  -", c["nombre"])

## 4. Corrección: el verificador rechaza las transformaciones ilegales

Las dos últimas candidatas son trampas habituales:

- **`A*B` elementwise**: multiplicar celda a celda. Es rapidísimo y *parece* «multiplicar matrices», pero
  no lo es: el producto matricial suma sobre una dimensión, no es el producto elemento a elemento.
- **`B@A`**: multiplicar en el orden cambiado. La multiplicación de matrices **no es conmutativa**, así
  que `B@A` casi nunca coincide con `A@B`.

Un agente que solo mirase el reloj caería en ellas (son veloces). El verificador, que **comprueba la
corrección primero**, las descarta. Veámoslo.

In [ ]:
print("Verificacion de las dos candidatas sospechosas:\n")
for c in catalogo:
    if c["esperado"] == "incorrecta":
        inf = verificador(c["nombre"], c["ejecuta"], referencia, tiempo_base)
        veredicto = "ACEPTADA" if inf["correcto"] else "RECHAZADA (resultado incorrecto)"
        print(f"  {c['nombre']:<40} -> {veredicto}")

print("\nLas dos quedan fuera: rapidas, si, pero calculan otra cosa.")

Esto es lo que separa un optimizador serio de uno ingenuo. La velocidad sin corrección no sirve: el
verificador actúa de **red de seguridad** y nos deja proponer ideas arriesgadas sin miedo, porque sabemos
que cualquier propuesta que rompa el resultado será descartada automáticamente.

## 5. Evaluar todo el catálogo

Pasamos cada candidato por el verificador y construimos una tabla con su veredicto y su speedup. Recuerda:
los números exactos dependen de tu máquina; lo que importa es el **patrón**. Verás que las dos incorrectas
se rechazan, que reordenar bucles da una mejora modesta y que **vectorizar con numpy dispara el speedup**.

In [ ]:
def evaluar_catalogo(A, B, reps=3):
    'Verifica todos los candidatos y devuelve la lista de informes con su speedup.'
    ref = A @ B
    cat = construye_catalogo(A, B)
    t_base = medir_tiempo(cat[0]["ejecuta"], reps)   # el primero es la base
    informes = []
    for c in cat:
        inf = verificador(c["nombre"], c["ejecuta"], ref, t_base, reps)
        inf["esperado"] = c["esperado"]
        informes.append(inf)
    return informes

informes = evaluar_catalogo(A, B)

print(f"{'candidato':<40}{'veredicto':<14}{'speedup':>10}")
print("-" * 64)
for inf in informes:
    if inf["correcto"]:
        print(f"{inf['nombre']:<40}{'correcto':<14}{inf['speedup']:>9.1f}x")
    else:
        print(f"{inf['nombre']:<40}{'RECHAZADO':<14}{'-':>10}")

legales = [inf for inf in informes if inf["correcto"]]
mejor = max(legales, key=lambda d: d["speedup"])
print(f"\nMejor transformacion legal: '{mejor['nombre']}' con ~{mejor['speedup']:.0f}x")

Tres lecciones de la tabla:

- Las **dos incorrectas** ni aparecen con speedup: se rechazan.
- **Reordenar bucles** (`ikj`) suele dar una mejora pequeña pero real respecto a la base.
- El gran salto llega al **delegar en numpy**: cuanto más trabajo dejamos en código vectorizado (bloques
  grandes, filas, o la operación entera), más rápido. El mejor está varias veces —a menudo cientos— por
  encima de la base. La cifra concreta es tuya; la tendencia es universal.

## 6. El agente: proponer, verificar, guardar la mejor, repetir

Ahora el bucle. El **agente** tiene una **política** que propone transformaciones. Empezamos con la
política más tonta posible, que **simula a un modelo que adivina a ciegas**: en cada iteración elige un
candidato al azar del catálogo. El verificador lo juzga; si es correcto y mejora la mejor marca, lo
guardamos. Si es incorrecto (o peor), se descarta y seguimos.

Como ya hemos medido el speedup real de cada candidato, ahora simulamos el bucle **consultando** esas
medidas (una propuesta incorrecta cuenta como speedup 0: rechazada). Así el bucle es instantáneo y podemos
repetirlo muchas veces para ver la tendencia media. Lo importante: el **mejor encontrado solo puede subir o
quedarse igual**, nunca empeora, porque siempre guardamos lo mejor visto.

In [ ]:
# Speedup de cada candidato segun lo ya verificado; las incorrectas valen 0 (se rechazan).
speedups = np.array([inf["speedup"] if inf["correcto"] else 0.0 for inf in informes])
nombres  = [inf["nombre"] for inf in informes]
M = len(speedups)

def bucle_aleatorio(n_iter, semilla):
    'Politica ciega: en cada paso propone un candidato al azar. Devuelve el mejor-hasta-ahora por iteracion.'
    rng = np.random.default_rng(semilla)
    mejor = 0.0
    curva = []
    traza = []
    for _ in range(n_iter):
        idx = rng.integers(M)                 # propuesta a ciegas
        sp = speedups[idx]
        acepta = sp > mejor                    # correcta Y mejor que lo que teniamos
        if acepta:
            mejor = sp
        traza.append((nombres[idx], sp, acepta))
        curva.append(mejor)
    return np.array(curva), traza

# Una corrida de ejemplo: vemos las primeras propuestas y el veredicto.
curva, traza = bucle_aleatorio(n_iter=30, semilla=1)
print("Primeras propuestas de una corrida (la politica va a ciegas):\n")
for nom, sp, acepta in traza[:8]:
    if sp == 0.0:
        estado = "rechazada (incorrecta)"
    elif acepta:
        estado = f"ACEPTADA, nuevo mejor {sp:.0f}x"
    else:
        estado = f"correcta pero peor ({sp:.0f}x)"
    print(f"  propone {nom:<40} -> {estado}")
print(f"\nMejor speedup tras 30 iteraciones: {curva[-1]:.0f}x")

In [ ]:
# Promediamos muchas corridas para ver la TENDENCIA: el mejor sube deprisa y luego se estanca.
n_iter = 30
corridas = np.array([bucle_aleatorio(n_iter, s)[0] for s in range(300)])
media = corridas.mean(axis=0)
mejor_posible = speedups.max()

plt.figure(figsize=(6.4, 3.8))
for s in range(8):                                  # unas pocas corridas individuales, en gris
    plt.step(range(1, n_iter + 1), corridas[s], where="post", color="0.8", lw=1)
plt.step(range(1, n_iter + 1), media, where="post", color="black", lw=2.2,
         label="media de 300 corridas")
plt.axhline(mejor_posible, color="red", ls="--", lw=1.3,
            label=f"mejor posible (~{mejor_posible:.0f}x)")
plt.xlabel("iteracion del bucle")
plt.ylabel("mejor speedup encontrado")
plt.title("El agente mejora y se estanca")
plt.legend(); plt.tight_layout(); plt.show()
print("El mejor sube rapido al principio y se aplana al acercarse al optimo.")

Esa curva —subir deprisa y **estancarse** al acercarse al óptimo— es la firma de cualquier optimizador por
búsqueda. Las propuestas malas o incorrectas no hacen daño: el verificador las filtra y la mejor marca se
mantiene. Lo único que el bucle necesita para funcionar es **un generador de ideas** (la política) y **un
juez honesto** (el verificador).

## 7. Best-of-N: probar varias veces y quedarse con la mejor

Si una sola propuesta es una apuesta, ¿por qué conformarse con una? **Best-of-N** consiste en pedir $N$
propuestas independientes y quedarse con la mejor que pase el verificador. Como nos quedamos con el
**máximo** de $N$ intentos en vez de con uno solo, el resultado típico es mejor.

Lo medimos comparando, sobre miles de simulaciones, el speedup medio de **una sola tirada** frente al de
**best-of-5**. (El libro reporta justo este efecto con un compilador real: la tirada única ronda 2,66x y
best-of-5 sube a 3,54x. Nuestros números serán otros porque aquí no hay compilador, pero la **dirección**
es la misma.)

In [ ]:
rng = np.random.default_rng(0)
trials = 4000

def best_of_n(n):
    'Speedup medio quedandote con la MEJOR de n propuestas a ciegas (incorrectas = 0).'
    picks = speedups[rng.integers(0, M, size=(trials, n))]
    return picks.max(axis=1).mean()

media_unica = best_of_n(1)        # tirada unica
media_5     = best_of_n(5)        # best-of-5
print(f"Tirada unica (best-of-1): speedup medio ~ {media_unica:.1f}x")
print(f"Best-of-5:                 speedup medio ~ {media_5:.1f}x")
print(f"Best-of-5 supera a la tirada unica: {media_5 > media_unica}")

In [ ]:
Ns = [1, 2, 3, 5, 8]
medias = [best_of_n(n) for n in Ns]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8.6, 3.6))

ax1.bar(["tirada\nunica", "best-of-5"], [media_unica, media_5],
        color=["0.7", "black"])
ax1.set_ylabel("speedup medio")
ax1.set_title("Una tirada vs best-of-5")
for i, v in enumerate([media_unica, media_5]):
    ax1.text(i, v, f"{v:.1f}x", ha="center", va="bottom")

ax2.plot(Ns, medias, "o-", color="black")
ax2.set_xlabel("N (numero de propuestas)")
ax2.set_ylabel("speedup medio (best-of-N)")
ax2.set_title("Mas intentos, mejor resultado")
plt.tight_layout(); plt.show()
print("Best-of-N crece con N, pero con rendimientos decrecientes: cada intento extra aporta menos.")

Best-of-N gana **siempre** a la tirada única (el máximo de varios nunca es menor que uno solo), pero con
**rendimientos decrecientes**: pasar de 1 a 2 intentos ayuda mucho; de 8 a 9, casi nada. Por eso en la
práctica se elige una $N$ pequeña (5 suele ser un buen punto). Es el mismo razonamiento que hay detrás de
muestrear varias respuestas de un LLM y quedarse con la mejor verificada.

## 8. La media geométrica: resumir mejoras sobre varios tamaños

Un speedup en un único tamaño de matriz dice poco. ¿Y si la transformación solo ayuda con matrices
grandes? Repetimos la búsqueda en **varios tamaños** y resumimos el mejor speedup de cada uno con la
**media geométrica**, que es la forma correcta de promediar factores de aceleración (un 4x y un 1/4x deben
cancelarse, y la media geométrica de 4 y 0,25 es 1; la aritmética daría 2,1, que engaña).

$$\text{media geométrica} = \left(\prod_{i} s_i\right)^{1/n}$$

In [ ]:
tamanos = [24, 40, 56, 72]
mejores = []
for n in tamanos:
    An = np.random.randn(n, n)
    Bn = np.random.randn(n, n)
    infs = evaluar_catalogo(An, Bn)
    sp_mejor = max(i["speedup"] for i in infs if i["correcto"])
    mejores.append(sp_mejor)
    print(f"  N={n:>3}: mejor speedup ~ {sp_mejor:>7.0f}x")

geo = np.exp(np.mean(np.log(mejores)))      # media geometrica = exp(media de los logaritmos)
arit = np.mean(mejores)
print(f"\nMedia geometrica del mejor speedup: ~{geo:.0f}x")
print(f"(La media aritmetica seria ~{arit:.0f}x, sesgada hacia el tamano mas favorable.)")

In [ ]:
plt.figure(figsize=(6.0, 3.6))
plt.bar([str(n) for n in tamanos], mejores, color="0.5")
plt.axhline(geo, color="red", ls="--", lw=1.4, label=f"media geometrica ~{geo:.0f}x")
plt.xlabel("tamano de la matriz (N)")
plt.ylabel("mejor speedup encontrado")
plt.title("El mejor speedup por tamano, y su media geometrica")
plt.legend(); plt.tight_layout(); plt.show()
print("Cuanto mas grande la matriz, mas castiga el Python puro y mayor es el margen de la vectorizacion.")

La media geométrica da **un solo número honesto** que resume cómo de buena es la mejor transformación a lo
largo de todo el rango de tamaños. Es exactamente así como se reportan los resultados en los artículos de
optimización de código: nunca un speedup suelto, sino la media geométrica sobre un conjunto de problemas.

## 9. De la política aleatoria al LLM: esto es COMPILOT

Hasta aquí la política era **ciega**: proponía candidatos al azar, con la misma probabilidad para una idea
mala que para una buena. Funciona, pero desperdicia muchos intentos. Una política **informada** —como un
LLM— no parte de cero: trae **conocimiento previo** (sabe que vectorizar suele ayudar) y, sobre todo, **lee
el feedback** del verificador. El efecto es que **concentra sus propuestas en la zona prometedora** en vez
de repartirlas por igual.

Comparamos las dos sobre el mismo catálogo, ordenado de menos a más vectorizado. La informada da más
probabilidad a las transformaciones más vectorizadas (las que el feedback señala como buenas); verás que su
mejor marca **sube antes** en cada iteración.

In [ ]:
# Eje estructural: de menos a mas vectorizado (correlaciona con la velocidad).
orden = ["bucle ijk (base, Python puro)", "bucle ikj (Python puro)",
         "bloques numpy b=2", "bloques numpy b=4", "bloques numpy b=8",
         "filas vectorizadas (numpy)", "bloques numpy b=16",
         "bloques numpy b=32", "numpy A@B (todo vectorizado)"]
sp_por_nombre = {inf["nombre"]: inf["speedup"] for inf in informes if inf["correcto"]}
sp_knob = np.array([sp_por_nombre[n] for n in orden])
K = len(sp_knob)

def politica_ciega(n_iter, semilla):
    'Reparte las propuestas por igual entre todos los candidatos (no sabe nada).'
    rng = np.random.default_rng(semilla)
    mejor = 0.0; curva = []
    for _ in range(n_iter):
        cand = rng.integers(K)
        mejor = max(mejor, sp_knob[cand])
        curva.append(mejor)
    return np.array(curva)

def politica_informada(n_iter, semilla, foco=2.0):
    'Como un LLM: concentra las propuestas en lo mas vectorizado, que es donde el feedback ve mejoras.'
    rng = np.random.default_rng(semilla)
    pesos = np.arange(1, K + 1, dtype=float) ** foco   # mas peso a lo mas vectorizado
    pesos /= pesos.sum()
    mejor = 0.0; curva = []
    for _ in range(n_iter):
        cand = rng.choice(K, p=pesos)                  # propuesta sesgada hacia la zona buena
        mejor = max(mejor, sp_knob[cand])
        curva.append(mejor)
    return np.array(curva)

n_iter = 15
ciega = np.array([politica_ciega(n_iter, s) for s in range(400)]).mean(axis=0)
lista = np.array([politica_informada(n_iter, s) for s in range(400)]).mean(axis=0)
print("Mejor speedup medio (media de 400 corridas):")
print(f"  tras 3 iteraciones  -> ciega ~{ciega[2]:.0f}x   informada ~{lista[2]:.0f}x")
print(f"  tras 15 iteraciones -> ciega ~{ciega[-1]:.0f}x   informada ~{lista[-1]:.0f}x")
print(f"La informada va por delante en cada iteracion: {bool(np.all(lista >= ciega - 1e-9))}")

In [ ]:
plt.figure(figsize=(6.4, 3.8))
plt.step(range(1, n_iter + 1), ciega, where="post", color="0.6", lw=2,
         label="politica ciega (adivina al azar)")
plt.step(range(1, n_iter + 1), lista, where="post", color="black", lw=2.2,
         label="politica informada (lee el feedback)")
plt.axhline(sp_knob.max(), color="red", ls="--", lw=1.2, label="optimo")
plt.xlabel("iteracion")
plt.ylabel("mejor speedup encontrado (media)")
plt.title("Una politica que lee el feedback converge antes")
plt.legend(); plt.tight_layout(); plt.show()
print("La informada se acerca antes al optimo: aprovecha lo que ya funciona en vez de empezar de cero.")

Y aquí está la idea entera del capítulo. **Cambiar la política aleatoria por un LLM que lee el feedback del
verificador es exactamente lo que hace un sistema como COMPILOT.** El montaje no cambia ni un milímetro:

- el **espacio de búsqueda** son transformaciones de código,
- el **verificador** es un compilador de verdad que comprueba que el programa transformado sigue siendo
  correcto y mide su tiempo real,
- la **política** ya no adivina: es un modelo que **lee el código, el feedback y los tiempos anteriores** y
  propone la siguiente transformación de forma informada,
- y se aplica **best-of-N** para quedarse con la mejor de varias propuestas verificadas.

Con ese montaje y un compilador real salen las cifras del libro: alrededor de **2,66x** de media en una sola
tirada y **3,54x** con best-of-5. Aquí, sin LLM ni compilador, has reproducido **el mecanismo**: proponer,
verificar, medir, quedarte con lo mejor. Lo demás es enchufar un modelo mejor en el sitio de la política.

## 10. Pruébalo tú

1. **Sube el tamaño.** Cambia `N = 64` a `N = 96` o `128` en la sección 1 y vuelve a ejecutar desde ahí.
   ¿Crece el speedup del mejor candidato? (El Python puro se castiga más con matrices grandes.)
2. **Rompe una transformación.** En `construye_catalogo`, añade un candidato que multiplique por bloques
   pero **se salte el último bloque** (por ejemplo cambia un `range(0, N, b)` por `range(0, N - b, b)`).
   El verificador debería **rechazarlo**. Compruébalo.
3. **Cambia la N de best-of-N.** En la sección 7, prueba `best_of_n(10)` o `best_of_n(20)`. ¿Cuánto más
   ganas? Fíjate en los rendimientos decrecientes.
4. **Sube la exploración.** En `politica_informada`, prueba `eps=0.5` y `eps=0.05`. ¿Demasiada exploración
   la vuelve casi ciega? ¿Demasiada poca la deja atascada si empieza en un mal sitio?

## 11. Errores comunes

- **Optimizar sin verificar la corrección.** El error más caro: quedarte con la versión más rápida sin
  comprobar que da el resultado bueno. `A*B` es velocísimo y está mal. **Primero corrección, luego velocidad.**
- **Medir mal el tiempo.** Una sola medición está llena de ruido (el sistema operativo, otros procesos). Por
  eso repetimos y nos quedamos con el **mínimo**, y descartamos la primera ejecución (calentamiento).
- **Confundir el mecanismo con la magnitud.** Aquí los speedups son enormes (Python puro contra numpy). Con
  un compilador real son más modestos (un puñado de veces). El bucle es el mismo; cambian los números.
- **Promediar speedups con la media aritmética.** Para factores de aceleración, la media correcta es la
  **geométrica**; la aritmética se deja arrastrar por el tamaño más favorable.
- **Creer que más intentos siempre compensan.** Best-of-N tiene rendimientos decrecientes y cada intento
  cuesta tiempo o dinero. Hay un punto a partir del cual no merece la pena.

## 12. Qué te llevas

- El poder de los agentes optimizadores no está en la genialidad de una propuesta, sino en el **bucle**:
  proponer, **verificar corrección**, **medir rendimiento**, guardar lo mejor, repetir.
- El **verificador** es el que manda. Permite proponer ideas arriesgadas sin peligro, porque cualquier
  transformación que rompa el resultado se rechaza automáticamente.
- El **mejor encontrado** sube deprisa y se estanca; **best-of-N** mejora la tirada única con rendimientos
  decrecientes; y la **media geométrica** resume las mejoras a lo largo de varios tamaños.
- Cambiar la **política aleatoria** por un **LLM que lee el feedback** convierte este juguete en **COMPILOT**:
  el mismo bucle, con un compilador real de verificador y un modelo como generador de ideas. De ahí salen
  las cifras del libro (alrededor de 2,66x, y 3,54x con best-of-5).

**Para seguir:** este patrón —generar, verificar, seleccionar— reaparece en todo el facsímil de agentes;
el siguiente capítulo lo lleva a tareas donde el verificador no es un compilador sino otro modelo o un
conjunto de pruebas.

---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*